# 01 · Preparação do dataset no Colab, sem API (Olist → classificação de aspectos)

**Tarefa:** dado uma avaliação em português, devolver o sentimento por aspecto em JSON:

```json
{"produto":"positivo","entrega":"negativo","atendimento":"ausente"}
```

Valores possíveis: `positivo`, `negativo`, `neutro`, `ausente` (aspecto não mencionado).

**Pipeline**
1. Baixa as avaliações do Olist e filtra/amostra (balanceado por nota).
2. Gera rótulos *silver* com um LLM maior **rodando na GPU do Colab** (Qwen2.5-7B-Instruct em 4-bit, few-shot). Não usa a API da Anthropic.
3. Separa train/val/test e exporta o **teste para revisão humana** (rótulo *gold*).
4. Mede a concordância LLM × humano e checa se não há vazamento entre os splits.
5. Gera o `data.zip` para o notebook 02.

**Execução:** Colab com **GPU T4** (*Ambiente de execução → Alterar tipo → T4 GPU*). Tudo roda neste notebook; os arquivos ficam em `/content` e somem quando a sessão acaba, então baixe o `data.zip` no fim.

In [ ]:
!pip -q install -U bitsandbytes accelerate kagglehub

In [ ]:
import os, torch
from pathlib import Path
assert torch.cuda.is_available(), "Sem GPU: Ambiente de execução > Alterar tipo de ambiente de execução > T4 GPU."
os.chdir("/content")
print(torch.__version__, torch.cuda.get_device_name(0))

2.11.0+cu130 Tesla T4


In [ ]:
import pandas as pd, numpy as np, json, re, os, time

RAW = Path("data/raw/olist_order_reviews_dataset.csv")
if not RAW.exists():
    import kagglehub
    RAW = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce")) / "olist_order_reviews_dataset.csv"
df = pd.read_csv(RAW)
print(df.shape)
df.head(3)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
(99224, 7)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24


In [ ]:
SEED = 42
df = df.dropna(subset=["review_comment_message"]).copy()
df["text"] = (df["review_comment_title"].fillna("") + ". " + df["review_comment_message"]) \
    .str.replace(r"\s+", " ", regex=True).str.strip(". ").str.strip()
df = df[df.text.str.len().between(25, 400)].drop_duplicates("text")

# amostra balanceada por nota para não ficar dominado por 5 estrelas
parts = [g.sample(min(len(g), 700), random_state=SEED) for _, g in df.groupby("review_score")]
sample = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
sample["id"] = sample.index
print(len(sample)); print(sample.review_score.value_counts().sort_index())

3500
review_score
1    700
2    700
3    700
4    700
5    700
Name: count, dtype: int64


## Rotulagem silver com Qwen2.5-7B-Instruct (local, grátis)

- Modelo de 7B em 4-bit (~6 GB), cabe na T4. Cada avaliação é um pedido separado com **4 exemplos few-shot** e decodificação determinística; os pedidos são processados em lotes de 16.
- Tempo estimado: **20 a 40 min para 3.000 avaliações** (estimativa; depende da GPU sorteada). Teste primeiro com `N_LABEL = 200`.
- Cada lote é gravado em `labels_cache.jsonl`. Se a sessão cair, reexecute e ele continua, **desde que o arquivo ainda exista**. Para sobreviver a quedas, aponte `CACHE` para o Drive (linha comentada na célula).
- `N_LABEL = 3000` reproduz a primeira execução no Colab. Se os ids rotulados forem os mesmos, o **split fica idêntico** e o seu CSV revisado de 300 linhas continua valendo (a célula de revisão confere isso). Com outro valor o split muda e é preciso revisar de novo.

In [ ]:
ASPECTS = ["produto", "entrega", "atendimento"]
LABELS = ["positivo", "negativo", "neutro", "ausente"]
LABELER_MODEL = "Qwen/Qwen2.5-7B-Instruct"
N_LABEL = 3000          # use 200 para um teste rápido
BS = 16                 # reduza para 8 se faltar memória
CACHE = "labels_cache.jsonl"
# Para persistir no Drive: from google.colab import drive; drive.mount("/content/drive")
# CACHE = "/content/drive/MyDrive/ifood_qlora/labels_cache.jsonl"   (crie a pasta antes)

GUIDE = """Você anota avaliações de clientes de e-commerce brasileiro.
Para cada avaliação, classifique o sentimento em três aspectos:
- produto: qualidade, funcionamento, descrição/foto, tamanho, estado do item recebido.
- entrega: prazo, atraso, rastreamento, item não chegou, entrega rápida, embalagem/transporte.
- atendimento: vendedor, SAC, suporte, troca, devolução, reembolso, comunicação.
Valores: "positivo", "negativo", "neutro" (menção sem polaridade clara ou mista equilibrada) ou "ausente" (aspecto não mencionado).
Regras: não infira o que não está no texto; use apenas o texto, não a nota.
Responda SOMENTE com JSON compacto: {"produto":"...","entrega":"...","atendimento":"..."}"""

FEWSHOT = [
    ("Chegou antes do prazo e o produto é exatamente como na foto. Recomendo!",
     {"produto": "positivo", "entrega": "positivo", "atendimento": "ausente"}),
    ("Comprei há 20 dias e até hoje não chegou. O vendedor não responde.",
     {"produto": "ausente", "entrega": "negativo", "atendimento": "negativo"}),
    ("Produto veio com defeito, mas a loja trocou rapidinho.",
     {"produto": "negativo", "entrega": "ausente", "atendimento": "positivo"}),
    ("Produto ok, nada demais.",
     {"produto": "neutro", "entrega": "ausente", "atendimento": "ausente"}),
]

def parse_labels(s):
    m = re.search(r"\{.*?\}", s, re.S)
    if not m: return None
    try: d = json.loads(m.group(0))
    except Exception: return None
    if set(d) != set(ASPECTS) or any(d[a] not in LABELS for a in ASPECTS): return None
    return d

done = {}
if os.path.exists(CACHE):
    for l in open(CACHE): d = json.loads(l); done[d["id"]] = d
todo = [r for r in sample.head(N_LABEL).to_dict("records") if r["id"] not in done]
print(f"{len(done)} em cache, {len(todo)} para rotular (modelo: {LABELER_MODEL})")

if todo:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from tqdm.auto import tqdm
    ltok = AutoTokenizer.from_pretrained(LABELER_MODEL); ltok.padding_side = "left"
    lm = AutoModelForCausalLM.from_pretrained(
        LABELER_MODEL, device_map={"": 0}, dtype=torch.float16,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                               bnb_4bit_compute_dtype=torch.float16)).eval()

    def build_prompt(text):
        msgs = [{"role": "system", "content": GUIDE}]
        for t, d in FEWSHOT:
            msgs += [{"role": "user", "content": t},
                     {"role": "assistant", "content": json.dumps(d, ensure_ascii=False, separators=(",", ":"))}]
        msgs.append({"role": "user", "content": text})
        return ltok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    @torch.no_grad()
    def label_batch(rows):
        enc = ltok([build_prompt(r["text"]) for r in rows], return_tensors="pt",
                   padding=True, add_special_tokens=False).to(lm.device)
        gen = lm.generate(**enc, max_new_tokens=40, do_sample=False, temperature=None, top_p=None,
                          top_k=None, repetition_penalty=1.0, pad_token_id=ltok.pad_token_id)
        outs = ltok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        res = []
        for r, o in zip(rows, outs):
            d = parse_labels(o)
            if d: res.append({"id": int(r["id"]), **d})
        return res

    todo.sort(key=lambda r: len(r["text"]))      # lotes de tamanho parecido = menos padding
    t0 = time.time()
    with open(CACHE, "a") as f:
        for i in tqdm(range(0, len(todo), BS)):
            for res in label_batch(todo[i:i + BS]):
                f.write(json.dumps(res, ensure_ascii=False) + "\n"); done[res["id"]] = res
            f.flush()
    print(f"Tempo: {(time.time() - t0) / 60:.1f} min")
    del lm; torch.cuda.empty_cache()

n_fail = sum(1 for r in sample.head(N_LABEL).to_dict("records") if r["id"] not in done)
print(f"{n_fail} avaliações sem rótulo válido (descartadas)")
if not done:
    raise RuntimeError("Nenhum rótulo foi gerado. Teste com N_LABEL = 20 e inspecione as saídas do modelo.")

lab = pd.DataFrame(list(done.values()))
data = sample.merge(lab, on="id")[["id", "text", "review_score"] + ASPECTS]
print(len(data), "rotuladas")
for a in ASPECTS: print(a, data[a].value_counts().to_dict())

0 em cache, 3000 para rotular (modelo: Qwen/Qwen2.5-7B-Instruct)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

  0%|          | 0/188 [00:00<?, ?it/s]

Tempo: 23.9 min
0 avaliações sem rótulo válido (descartadas)
3000 rotuladas
produto {'negativo': 1110, 'positivo': 933, 'ausente': 682, 'neutro': 275}
entrega {'ausente': 1303, 'negativo': 864, 'positivo': 773, 'neutro': 60}
atendimento {'ausente': 2351, 'negativo': 394, 'positivo': 163, 'neutro': 92}


In [ ]:
# Splits: 2400 / 300 / 300 (ajusta sozinho se houver menos linhas rotuladas)
data = data.sample(frac=1, random_state=SEED).reset_index(drop=True)
n = len(data); n_test = n_val = min(300, n // 10)
test, val, train = data[:n_test], data[n_test:n_test + n_val], data[n_test + n_val:]

os.makedirs("data", exist_ok=True)
for name, d in [("train", train), ("val", val), ("test", test)]:
    d.to_json(f"data/{name}.jsonl", orient="records", lines=True, force_ascii=False)
test.to_csv("data/test_para_revisao.csv", index=False)
open("data/labeler.txt", "w").write(LABELER_MODEL + " (4-bit, few-shot)")
print(len(train), len(val), len(test))
if len(test) < 150:
    print(f"ATENÇÃO: o teste ficou com {len(test)} linhas, pequeno demais para avaliar. Aumente N_LABEL (3000 recomendado; mínimo ~1500) e refaça a rotulagem e o split.")

# Baixa o CSV para abrir no Google Sheets / Excel
from google.colab import files
files.download("data/test_para_revisao.csv")

2400 300 300


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Revisão humana do teste (gold)

1. O `test_para_revisao.csv` foi baixado pela célula anterior. Importe no Google Sheets (*Arquivo → Importar → Upload*) ou abra no Excel.
2. Corrija as colunas `produto`, `entrega`, `atendimento` onde o LLM errou. Valores válidos: `positivo`, `negativo`, `neutro`, `ausente`. Não altere a coluna `id` nem o texto. Revise pelo menos 150 linhas, idealmente todas.
3. Baixe como **CSV** (*Arquivo → Fazer download → CSV*).
4. Rode a célula abaixo e envie o CSV. Ela aceita separador `;`, alinha as linhas pelo `id` e confere se o arquivo pertence a **este** split.

**Se você já revisou 300 linhas numa execução anterior** com `N_LABEL = 3000`, envie esse mesmo arquivo: se os ids baterem, ele vale. Se não bater, a célula mostra quantos ids coincidem e você precisa revisar o novo `test_para_revisao.csv`.

In [ ]:
import io
from google.colab import files

orig = pd.read_csv("data/test_para_revisao.csv")
up = files.upload()                                   # selecione o CSV revisado
name, content = next(iter(up.items()))

def read_any(raw):
    for enc in ("utf-8-sig", "latin-1"):
        try: return pd.read_csv(io.BytesIO(raw), sep=None, engine="python", encoding=enc)
        except UnicodeDecodeError: continue

rev = read_any(content)
need = {"id", "text", *ASPECTS}
assert need <= set(rev.columns), f"Colunas faltando: {need - set(rev.columns)}. Colunas do arquivo: {list(rev.columns)}"

common = set(rev["id"]) & set(orig["id"])
if len(rev) != len(orig) or len(common) != len(orig):
    raise AssertionError(
        f"O arquivo '{name}' tem {len(rev)} linhas e o teste atual tem {len(orig)}; ids em comum: {len(common)}. "
        "Eles vêm de splits diferentes. Use N_LABEL = 3000 (mesmo da execução em que você revisou) ou revise o novo test_para_revisao.csv.")

rev = rev.set_index("id").loc[orig["id"]].reset_index()         # alinha a ordem pelo id
for a in ASPECTS: rev[a] = rev[a].astype(str).str.strip().str.lower()
bad = {a: sorted(set(rev[a]) - set(LABELS)) for a in ASPECTS}
assert not any(bad.values()), f"Valores inválidos no CSV revisado: {bad}"
rev.to_csv("data/test_revisado.csv", index=False)

changed = int((rev[ASPECTS].values != orig[ASPECTS].values).any(axis=1).sum())
print(f"Salvo em data/test_revisado.csv. Linhas alteradas em relação ao LLM: {changed} de {len(rev)}")
if changed == 0:
    print("ATENÇÃO: nenhuma linha foi alterada. Se você já corrigiu, confira se enviou o arquivo certo.")

Saving test_revisado.csv to test_revisado.csv
Salvo em data/test_revisado.csv. Linhas alteradas em relação ao LLM: 0 de 300
ATENÇÃO: nenhuma linha foi alterada. Se você já corrigiu, confira se enviou o arquivo certo.


In [ ]:
from sklearn.metrics import cohen_kappa_score, accuracy_score
silver = pd.read_csv("data/test_para_revisao.csv")
gold = pd.read_csv("data/test_revisado.csv")
assert len(silver) == len(gold) and (silver.id.values == gold.id.values).all(), (
    f"O CSV revisado ({len(gold)} linhas) não corresponde ao test_para_revisao.csv atual ({len(silver)} linhas); "
    f"ids em comum: {len(set(silver.id) & set(gold.id))}.")
for a in ASPECTS: gold[a] = gold[a].astype(str).str.strip().str.lower()

kappas = {}
for a in ASPECTS:
    kappas[a] = cohen_kappa_score(gold[a], silver[a])
    print(f"{a:12s} acordo={accuracy_score(gold[a], silver[a]):.3f}  kappa={kappas[a]:.3f}")
if all(accuracy_score(gold[a], silver[a]) == 1.0 for a in ASPECTS):
    print("ATENÇÃO: concordância perfeita em tudo. Isso significa que nenhuma linha foi corrigida (ou que os dois arquivos são o mesmo). "
          "Nesse caso o kappa não mede nada: não o use no README.")

gold.to_json("data/test.jsonl", orient="records", lines=True, force_ascii=False)
print(gold[ASPECTS].apply(pd.Series.value_counts).fillna(0).astype(int))

produto      acordo=1.000  kappa=1.000
entrega      acordo=1.000  kappa=1.000
atendimento  acordo=1.000  kappa=1.000
ATENÇÃO: concordância perfeita em tudo. Isso significa que nenhuma linha foi corrigida (ou que os dois arquivos são o mesmo). Nesse caso o kappa não mede nada: não o use no README.
          produto  entrega  atendimento
ausente        67      134          238
negativo      105       87           38
neutro         23        2           10
positivo      105       77           14


In [ ]:
# Sanidade: treino, validação e teste não podem compartilhar avaliações
tr, va, te = [pd.read_json(f"data/{s}.jsonl", lines=True) for s in ["train", "val", "test"]]
print("linhas:", len(tr), len(va), len(te))
ov_id = (set(tr.id) & set(te.id)) | (set(va.id) & set(te.id)) | (set(tr.id) & set(va.id))
ov_tx = (set(tr.text) & set(te.text)) | (set(va.text) & set(te.text)) | (set(tr.text) & set(va.text))
print("ids repetidos entre splits:", len(ov_id), "| textos repetidos:", len(ov_tx))
assert not ov_id and not ov_tx, "Há vazamento entre os splits. Refaça o split a partir do mesmo conjunto rotulado."
print("OK: splits disjuntos")

linhas: 2400 300 300
ids repetidos entre splits: 0 | textos repetidos: 0
OK: splits disjuntos


## Levar os dados para o notebook 02

A célula abaixo cria o `data.zip` (`train`, `val`, `test`, `labeler.txt`) e o baixa. No notebook 02, a primeira célula pede esse arquivo.

In [ ]:
import zipfile
with zipfile.ZipFile("data.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["train.jsonl", "val.jsonl", "test.jsonl", "labeler.txt"]:
        z.write(f"data/{name}", arcname=f"data/{name}")
print("Criado:", Path("data.zip").resolve(), f"({Path('data.zip').stat().st_size/1e3:.0f} KB)")
files.download("data.zip")

# Opcional: guardar também no Drive
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/ifood_qlora
# !cp -r data /content/drive/MyDrive/ifood_qlora/
# !cp labels_cache.jsonl /content/drive/MyDrive/ifood_qlora/

Criado: /content/data.zip (133 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>